In [ ]:
instructions
u nede to run this whole notebook in-order to work ur part properly:

01. copy this notebook
02. run whole notebook
03. save the results, then remove other ppls parts.
04. save it!
05. make sure to run the code several times
06. push the code to your relevent branch named as [preprocessing/IT2510XXXX] (if theres no any branche sunder ur it number, create one!)

In [ ]:
## importing necessary packages
# type this before type anything on ur notebook

import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/diabetes_binary_health_indicators_BRFSS2015.csv' ## upload and paste ur original dataset link like this, then


## Data Preprocessing: Null Value Check and Outlier Visualization, dropping y, train test split ----> chandira

First, let's check for any missing values in the dataset.

In [ ]:
print('Checking for null values:')
print(df.isnull().sum())

Next, we will visualize potential outliers using box plots for numerical features. We'll iterate through numerical columns to create these plots.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select only numerical columns for box plots
numerical_cols = df.select_dtypes(include=['number']).columns

# Determine the number of rows and columns for the subplot grid
num_features = len(numerical_cols)
num_rows = (num_features + 2) // 3 # Roughly 3 plots per row
num_cols = 3

plt.figure(figsize=(num_cols * 5, num_rows * 4))

for i, col in enumerate(numerical_cols):
    plt.subplot(num_rows, num_cols, i + 1)
    sns.boxplot(y=df[col])
    plt.title(f'Box Plot of {col}')
    plt.ylabel('')
    plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
## importing necessary packages

import pandas as pd
from google.colab import drive
from sklearn.model_selection import train_test_split

drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/datasets/diabetes_binary_health_indicators_BRFSS2015.csv'


In [ ]:
# Train test split, dropping Y (label)
df = pd.read_csv(file_path)
x= df.drop('Diabetes_binary', axis=1)
y=df['Diabetes_binary']

xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, stratify = y, random_state=42)

print("train : ", xtrain.shape)
print("test : ", xtest.shape)

print("train ratio : ")
print(ytrain.value_counts(normalize=True))
print("test ratio : ")
print(ytest.value_counts(normalize=True))

In [ ]:
## download splitted datasets

from google.colab import files

trainbalanced= pd.DataFrame(xtrain, columns = x.columns)
trainbalanced['Diabetes_binary']= ytrain

testrealdf = pd.DataFrame(xtest, columns = x.columns)
testrealdf['Diabetes_binary']= ytest

trainbalanced.to_csv('train.csv', index=False)
testrealdf.to_csv('testreal.csv', index=False)

files.download('train.csv')
files.download('testreal.csv')

initial correlation matrix, MI score ---> salman

Mutual Information measures the dependency between the variables. It is zero if and only if two random variables are independent, and higher values mean higher dependency.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the correlation matrix
corr_matrix = df.corr()

# Plotting the heatmap
plt.figure(figsize=(16, 12))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Features')
plt.show()


from sklearn.feature_selection import mutual_info_classif

# Calculate Mutual Information scores between features and the target variable 'Diabetes_binary'
# We use the original 'df' with the target column for MI calculation

# Ensure all features are numerical for MI calculation
# If there are categorical features, they should ideally be one-hot encoded or handled appropriately
# For this dataset, assuming all relevant features are numerical or can be treated as such for MI

x_mi = df.drop('Diabetes_binary', axis=1)
y_mi = df['Diabetes_binary']

# Convert all features to numeric, coercing errors to NaN and then filling them if necessary
# This step is crucial if the data contains non-numeric types after initial load
for col in x_mi.columns:
    if x_mi[col].dtype == 'object' or x_mi[col].dtype == 'category':
        # For simplicity, convert categorical objects to numerical using factorize or one-hot encode
        # Here, assuming they are binary or can be directly converted for MI_classif
        x_mi[col] = pd.factorize(x_mi[col])[0]
    x_mi[col] = pd.to_numeric(x_mi[col], errors='coerce')

# Fill any NaN values that might have been introduced by coercion for MI calculation
x_mi = x_mi.fillna(x_mi.mean())


mi_scores = mutual_info_classif(x_mi, y_mi, random_state=42)
mi_series = pd.Series(mi_scores, index=x_mi.columns)

# Sort the scores for better visualization
mi_series = mi_series.sort_values(ascending=False)

# Plotting the MI scores
plt.figure(figsize=(12, 8))
sns.barplot(x=mi_series.values, y=mi_series.index, palette='viridis')
plt.title('Mutual Information Scores with Diabetes_binary')
plt.xlabel('Mutual Information Score')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()


categorical vs numerical split + chi square + re correlation ----> thirshe

In [ ]:
# [cell 06]

from sklearn.feature_selection import SelectKBest, chi2

categorical_features = [
    'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
    'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare',
    'NoDocbcCost', 'DiffWalk', 'Sex', 'Education', 'Income'
]

# Make sure all selected categorical features exist in pdx
# Also ensure the target pduse is suitable for chi2 (integer/binary)

# Filter pdx to only include the identified categorical features
pdx_categorical = pdx[categorical_features].copy()

# Convert columns to appropriate type if necessary (e.g., int for chi2)
# Chi2 works best with non-negative integers. Our current float64 (0.0, 1.0, etc.) will work.
# Ensure target is also int/binary for chi2
pduse_chi2 = pduse.astype(int)

# Apply SelectKBest with chi2
# We'll select all categorical features to get their scores and p-values
selector_chi2 = SelectKBest(chi2, k='all')
selector_chi2.fit(pdx_categorical, pduse_chi2)

# Get the scores and p-values
chi2_scores = pd.DataFrame({
    'Feature': pdx_categorical.columns,
    'Chi2_Score': selector_chi2.scores_,
    'P_Value': selector_chi2.pvalues_
})

# Sort by Chi2 score in descending order to see the most important features first
chi2_scores = chi2_scores.sort_values(by='Chi2_Score', ascending=False)

print("Chi-Square Scores for Categorical Features (ranked by importance):\n")
display(chi2_scores)

print("\n--- Interpretation ---\n")
print("Higher Chi2_Score indicates a stronger statistical dependence between the feature and the target variable.")
print("Lower P_Value (typically < 0.05) suggests that the relationship is statistically significant.")
print("Features with high Chi2_Score and low P_Value are strong candidates for retention.")


In [ ]:
# Calculate the correlation matrix for the features in pdx
re_corr = pdx.corr()

print("Correlation Matrix / after chi square ")
display(re_corr.head())

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(18, 15))
sns.heatmap(inter_feature_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix / after chi square ')
plt.show()



feature dropping, re correlation -----> Kasundi

In [ ]:

#dropping features
features_to_drop = [
    'NoDocbcCost', 'Fruits', 'Veggies', 'Sex',
    'AnyHealthcare', 'CholCheck'
]


# 'pdx' ---> original datafarme foem traiinset
# .copy() ---> araksawata use krnne, nttm main dataframe eka edit wenn pluwn!!!
# 'errors='ignore''column ekak hmbune nththm err ek ignore krnw
pdx_after_drop = pdx.drop(columns=features_to_drop, errors='ignore').copy()

pdx_after_drop_corr = pdx_after_drop.corr()

print("Corrected Inter-feature Correlation Matrix (first 5 rows/cols after dropping features):")
display(pdx_after_drop_corr.head())

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(18, 15))
sns.heatmap(pdx_after_drop_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Inter-feature Correlation Heatmap (after correctly removing features)')
plt.show()

print(f"\nbefore dropping: {pdx.shape[1]}")
print(f"attempted to drop: {features_to_drop}")
print(f"Features after dropping: {pdx_after_drop.shape[1]}")
print(f"Remaining features: {list(pdx_after_drop.columns)}")



numerical & categorical data encoding -----> Gayashi

notes: Numerical Features: We will use StandardScaler to standardize them.
Categorical Features: We will use OneHotEncoder to convert them into a binary matrix.



In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify numerical & categorical features ---> pdx_after_drop

# remaining in pdx_after_drop:
# ['HighBP', 'HighChol', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
#  'PhysActivity', 'HvyAlcoholConsump', 'GenHlth', 'MentHlth', 'PhysHlth',
#  'DiffWalk', 'Age', 'Education', 'Income']

numerical_features = [
    'BMI', 'GenHlth', 'MentHlth', 'PhysHlth', 'Age', 'Income'
]
categorical_features_for_ohe = [
    'HighBP', 'HighChol', 'Smoker', 'Stroke', 'HeartDiseaseorAttack',
    'PhysActivity', 'HvyAlcoholConsump', 'DiffWalk', 'Education'
]

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_for_ohe)
    ])

pdx_processed = preprocessor.fit_transform(pdx_after_drop)


# get_feature_names_out method:  returns the column names in the order they appear from the numerical and one-hot encoded features.

# The order of the feature names must match the order from preprocessor
# get numerical feature names first, next categorical feature names
feature_names = numerical_features + \
                list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features_for_ohe))

# Convert processed array --> DataFrame
pdx_final_df = pd.DataFrame(pdx_processed, columns=feature_names, index=pdx_after_drop.index)

print("Original features (first 5 rows of numerical and categorical):")
display(pdx_after_drop[numerical_features + categorical_features_for_ohe].head())

print("\nProcessed features (first 5 rows of scaled numerical and one-hot encoded categorical):")
display(pdx_final_df.head())

print(f"\nNumber of features before processing: {pdx_after_drop.shape[1]}")
print(f"Number of features after processing and one-hot encoding: {pdx_final_df.shape[1]}")

undersampling, xport ----> Mithsuka

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from google.colab import files



# random_state for reproducibility
rus = RandomUnderSampler(random_state=42)

print(f"shape of features before undersampling: {pdx_final_df.shape}")
print(f"sahpe of target target before undersampling: {pduse.shape}")
print(f"class distribution before undersampling:\n{pduse.value_counts()}")

# Apply undersampling to the processed features and original target variable
# pdx_final_df ---> processed training features
# pduse ---> original training target labels
X_resampled, y_resampled = rus.fit_resample(pdx_final_df, pduse)

print(f"\nfeatures - after undersampling: {X_resampled.shape}")
print(f"taregt - after undersampling: {y_resampled.shape}")
print(f"class distribution after undersampling:\n{y_resampled.value_counts()}")

# Display the first few rows of the resampled features
print("\nFirst 5 rows of resampled features:")
display(X_resampled.head())


# X_resampled
features_filename = 'undersampled_features.csv'
X_resampled.to_csv(features_filename, index=False)
print(f"'{features_filename}'")
files.download(features_filename)

# y_resampled
target_filename = 'undersampled_target.csv'
y_resampled.to_csv(target_filename, index=False)
print(f"'{target_filename}'")
files.download(target_filename)

# test set preprocessing
# 1. Drop the same features from xtest
xtest_after_drop = xtest.drop(columns=features_to_drop, errors='ignore').copy()

# 2. Transform the test set using the preprocessor fitted on training data
xtest_processed = preprocessor.transform(xtest_after_drop)

# 3. Convert to DataFrame for consistency
xtest_final_df = pd.DataFrame(xtest_processed, columns=feature_names, index=xtest.index)

print(f"Original xtest shape: {xtest.shape}")
print(f"Processed xtest shape: {xtest_final_df.shape}")

print("\nFirst 5 rows of processed test features:")
display(xtest_final_df.head())

## downloading xtest (processed file)
from google.colab import files

# Save the preprocessed test features to a CSV file
test_features_filename = 'preprocessed_xtest.csv'
xtest_final_df.to_csv(test_features_filename, index=False)
print(f"Saved '{test_features_filename}'")

# Download the file
files.download(test_features_filename)